# Configure and inspect a completion workflow

Use the same synchronous workflow in an application or a running Jupyter kernel.
This notebook uses fixed text callbacks offline. It tests integration mechanics,
not provider support, model learning or mitigation effectiveness. MCS remains
unavailable because the fixtures supply no token log probabilities.

These APIs require the development checkout and are not included in v0.5.0.
Select its installed Python environment as this notebook's kernel. Open this
notebook from `notebooks/`; the relative path below loads the companion
application's fixture functions without executing its `main()`.


In [ ]:
%run -n ../../examples/completion_workflow.py

import sys
print("Kernel interpreter:", sys.executable)

## Choose the target and features

The callback protocol is explicit, with no HTTP endpoint or credential. Change
`ensemble_draws` and the matching `EnsembleSpec` together. Select `rewriting=False`
to use original evidence; that choice creates no auxiliary clients. For a single
answer choose one draw and pass `ensemble=None`.


In [ ]:
config = fixture_config()
features = FeatureChoices(mia_scoring=False, ensemble_draws=3, rewriting=True)
ensemble = EnsembleSpec(draws=features.ensemble_draws, min_parsed=3,
                        max_workers=1, retain_draws=True)
budget = CallBudget(12)
workflow = make_fixture_workflow(config, features, budget)

local = config.inspect(features)  # Local declarations, without calls.
print(config.model, config.protocol.value, config.endpoint, dict(config.parameters))
print("Unsupported:", local.unsupported, "Needs probe:", local.requires_probe)

## Inspect readiness, then run

Preflight exercises only the selected fixture roles and consumes admission slots.
One budget covers their probes, rewriting, verification and target draws. In a
real callback every physical request must pass through the supplied admission
hook with hidden retries disabled. Cooperation is not provider attestation.


In [ ]:
preflight = workflow.preflight(rewrite=features.rewriting, ensemble=ensemble)
print("Ready:", preflight.ready_for_batch, preflight.affected_role, preflight.error)
if not preflight.ready_for_batch:
    raise RuntimeError(preflight.error)

accepted = workflow.run(fixture_item(), rewrite=features.rewriting, ensemble=ensemble)
print(json.dumps(result_summary(accepted), indent=2))

Agreement is the share of usable draws supporting the selected answer. It
does not estimate memorization. The constant `up` answer and positive verifier
verdict are fixtures. Automated fidelity acceptance is a check result, not a
guarantee. `p_memorized` stays `None`; read `mcs_unavailable_reason` before using it.
When rewriting is selected, all target draws share the single accepted rewrite
(`shared_per_item`). Otherwise they use the original evidence.


## Inspect a failed intervention

This second fixture runs only when rewriting is selected. It changes `2%` to `3%`.
The deterministic fidelity gate rejects
it before target draws. The workflow retains the candidate and error, and does
not substitute an original-evidence answer. Its role probes share the same budget.


In [ ]:
rejected = None
if features.rewriting:
    rejected_workflow = make_fixture_workflow(
        config, features, budget, rewritten_evidence=REJECTED_EVIDENCE,
    )
    rejected = rejected_workflow.run(fixture_item(), rewrite=features.rewriting, ensemble=ensemble)
    print(json.dumps(result_summary(rejected), indent=2))
print("Admitted fixture operations:", budget.used, "of", budget.limit,
      "(zero HTTP requests)")

## Separate configuration errors from callback runtime errors

Native Anthropic validation below rejects `temperature` and MIA/MCS locally,
before client construction or credential lookup. The final example deliberately
returns an awaitable from a callback: its `callback_runtime_error` is a violation
of the synchronous callback contract, independent of model parameter support.
Use a synchronous client directly in this kernel; do not call `asyncio.run`.


In [ ]:
show_local_native_errors()
show_callback_runtime_error()

## Read and replay the measured evidence

The [engineering guide](../guides/completion-api.md) covers explicit live settings,
callback integration and genuine logprob scoring with matched calibration.
The [2026-10-31 pilot](../results/2026-10-31-completion-api-pilot.md) found no
mitigation benefit: all 16 rewrites were rejected, leaving zero valid rewrite
pairs. Fresh calibration and full two-mode live coverage remain incomplete.

From the repository root, this command recomputes the saved paired metrics offline:

```text
uv run --no-sync python scripts/run_intervention.py --replay docs/results/2026-10-31-completion-api/anthropic-evaluation-live
```

Replay verifies saved metrics; it makes no fresh inference or calibration calls.
The dated study's eligibility budget is exhausted. A new experiment needs its own
declared protocol, budget and inputs.
